In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

# Harness-only conversion helpers: the mined fixture is stored in target-side
# form, while the oracle must receive the semantically equivalent pandas form.
def _to_pandas_fixture(value):
    if isinstance(value, pl.DataFrame):
        return value.to_pandas()
    if isinstance(value, pl.Series):
        return value.to_pandas()
    if isinstance(value, list):
        return [_to_pandas_fixture(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_to_pandas_fixture(item) for item in value)
    if isinstance(value, dict):
        return {key: _to_pandas_fixture(item) for key, item in value.items()}
    if isinstance(value, SimpleNamespace):
        return SimpleNamespace(**{
            key: _to_pandas_fixture(item) for key, item in vars(value).items()
        })
    return value

def _to_polars_fixture(value):
    if isinstance(value, pd.DataFrame):
        return pl.from_pandas(value)
    if isinstance(value, pd.Series):
        return pl.from_pandas(value)
    if isinstance(value, list):
        return [_to_polars_fixture(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_to_polars_fixture(item) for item in value)
    if isinstance(value, dict):
        return {key: _to_polars_fixture(item) for key, item in value.items()}
    if isinstance(value, SimpleNamespace):
        return SimpleNamespace(**{
            key: _to_polars_fixture(item) for key, item in vars(value).items()
        })
    return value


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- dedup_nested_cross_join ---
FIX_DEDUP_NESTED_CROSS_JOIN_DF_CELLS = pl.DataFrame({"x1":[10,50,100],"y1":[10,10,50],"x2":[45,90,145],"y2":[45,45,90],"width":[35,40,45],"height":[35,35,40],"area":[1225,1400,1800]})
FIX_DEDUP_NESTED_CROSS_JOIN_DF_CELLS_CP = pl.DataFrame({"index_":[0,1,2],"x1_":[10,50,100],"y1_":[10,10,50],"x2_":[45,90,145],"y2_":[45,45,90],"width_":[35,40,45],"height_":[35,35,40],"area_":[1225,1400,1800]})

# --- dedup_redundant_filter ---
FIX_DEDUP_REDUNDANT_FILTER_DF_CELLS = pl.DataFrame({"index_":[0,1,2],"x1":[10,50,100],"y1":[10,10,50],"x2":[45,90,145],"y2":[45,45,90],"width":[35,40,45],"height":[35,35,40],"area":[1225,1400,1800]})
FIX_DEDUP_REDUNDANT_FILTER_DF_CROSS_CELLS = pl.DataFrame({"index":[0,0],"x1":[10,10],"y1":[10,10],"x2":[45,45],"y2":[45,45],"width":[35,35],"height":[35,35],"area":[1225,1225],"index_":[1,2],"x1_":[50,100],"y1_":[10,50],"x2_":[90,145],"y2_":[45,90],"width_":[40,45],"height_":[35,40],"area_":[1400,1800],"x_left":[45,45],"x_right":[50,100],"y_top":[10,10],"y_bottom":[45,45],"overlapping_x":[0,0],"overlapping_y":[35,0],"diff_x":[5,55],"diff_y":[0,5],"int_area":[0,0],"contained":[False,False],"adjacent":[True,False],"redundant":[False,False]})

# --- dedup_vertical_sort_cumcount ---

print("✅ Fixtures loaded")
OCRDataframe = SimpleNamespace  # mock for testing


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_dedup_nested_cross_join(df_cells, df_cells_cp):
    df_cross_cells = df_cells.reset_index().merge(df_cells_cp, how='cross')
    df_cross_cells = df_cross_cells[df_cross_cells["index"] != df_cross_cells["index_"]]
    df_cross_cells = df_cross_cells[df_cross_cells["area"] <= df_cross_cells["area_"]]
    return df_cross_cells

def before_dedup_redundant_filter(df_cells, df_cross_cells):
    redundant_cells = df_cross_cells[df_cross_cells["redundant"]]['index_'].drop_duplicates().values.tolist()
    df_final_cells = df_cells.drop(labels=redundant_cells)
    return df_final_cells

def before_dedup_vertical_sort_cumcount(df_cells=None):
    if df_cells is None:
        df_cells = pd.DataFrame({"x1":[0],"x2":[10],"y1":[0],"y2":[10],"content":["a"]})
    df_cells = df_cells.sort_values(by=["x1", "x2", "y1", "y2"])
    df_cells["cell_rk"] = df_cells.groupby(["x1", "x2", "y1"]).cumcount()
    df_cells = df_cells[df_cells["cell_rk"] == 0]
    return df_cells

_oracle_dedup_redundant_filter = before_dedup_redundant_filter
def before_dedup_redundant_filter(*args, **kwargs):
    return _oracle_dedup_redundant_filter(
        *[_to_pandas_fixture(value) for value in args],
        **{key: _to_pandas_fixture(value) for key, value in kwargs.items()},
    )


In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_dedup_nested_cross_join(df_cells, df_cells_cp):
    df_cross_cells = df_cells.with_row_index("index").join(df_cells_cp, how="cross")
    df_cross_cells = df_cross_cells.filter(pl.col("index") != pl.col("index_"))
    df_cross_cells = df_cross_cells.filter(pl.col("area") <= pl.col("area_"))
    return df_cross_cells

def gen_dedup_redundant_filter(df_cells, df_cross_cells):

    redundant_cells = (
        df_cross_cells.filter(pl.col("redundant"))["index_"].unique(maintain_order=True).to_list()
    )
    df_final_cells = df_cells.filter(~pl.col("index_").is_in(redundant_cells))
    return df_final_cells

def gen_dedup_vertical_sort_cumcount(df_cells=None):
    if df_cells is None:
        df_cells = pl.DataFrame({"x1":[0],"x2":[10],"y1":[0],"y2":[10],"content":["a"]})

    df_cells = df_cells.sort(["x1", "x2", "y1", "y2"])
    df_cells = df_cells.with_row_index("_row_nr").with_columns(
        (pl.col("_row_nr").cum_count().over(["x1", "x2", "y1"]) - 1).alias("cell_rk")
    ).drop("_row_nr")
    df_cells = df_cells.filter(pl.col("cell_rk") == 0)
    return df_cells

def _to_pandas_fixture(obj):
    if isinstance(obj, pl.Series):
        return obj.to_pandas()
    if isinstance(obj, pl.DataFrame):
        return obj.to_pandas()
    if isinstance(obj, pl.LazyFrame):
        return obj.collect().to_pandas()
    if isinstance(obj, list):
        return [_to_pandas_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_pandas_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_pandas_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), (pl.DataFrame, pl.LazyFrame)):
        return SimpleNamespace(df=_to_pandas_fixture(obj.df))
    return obj

def _to_polars_fixture(obj):
    if isinstance(obj, pd.Series):
        return pl.Series(obj.name or "series", obj.to_list())
    if isinstance(obj, pd.DataFrame):
        return pl.from_pandas(obj)
    if isinstance(obj, list):
        return [_to_polars_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_polars_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_polars_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), pd.DataFrame):
        return SimpleNamespace(df=_to_polars_fixture(obj.df))
    return obj

def _wrap_before_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_pandas_fixture(self.df)
        try:
            return fn(*[_to_pandas_fixture(a) for a in args], **{k: _to_pandas_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

def _wrap_gen_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_polars_fixture(self.df)
        try:
            return fn(*[_to_polars_fixture(a) for a in args], **{k: _to_polars_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

for _name, _fn in list(globals().items()):
    if callable(_fn) and _name.startswith("before_"):
        globals()[_name] = _wrap_before_func(_fn)
    elif callable(_fn) and _name.startswith("gen_"):
        globals()[_name] = _wrap_gen_func(_fn)

# ── Test harness type adapters ─────────────────────────────────────────────
def _to_pandas_fixture(obj):
    if isinstance(obj, pl.Series):
        return obj.to_pandas()
    if isinstance(obj, pl.DataFrame):
        return obj.to_pandas()
    if isinstance(obj, pl.LazyFrame):
        return obj.collect().to_pandas()
    if isinstance(obj, list):
        return [_to_pandas_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_pandas_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_pandas_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), (pl.DataFrame, pl.LazyFrame)):
        return SimpleNamespace(df=_to_pandas_fixture(obj.df))
    return obj

def _to_polars_fixture(obj):
    if isinstance(obj, pd.Series):
        return pl.Series(obj.name or "series", obj.to_list())
    if isinstance(obj, pd.DataFrame):
        return pl.from_pandas(obj)
    if isinstance(obj, list):
        return [_to_polars_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_polars_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_polars_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), pd.DataFrame):
        return SimpleNamespace(df=_to_polars_fixture(obj.df))
    return obj

def _wrap_before_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_pandas_fixture(self.df)
        try:
            return fn(*[_to_pandas_fixture(a) for a in args], **{k: _to_pandas_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

def _wrap_gen_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_polars_fixture(self.df)
        try:
            return fn(*[_to_polars_fixture(a) for a in args], **{k: _to_polars_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

for _name, _fn in list(globals().items()):
    if callable(_fn) and _name.startswith("before_"):
        globals()[_name] = _wrap_before_func(_fn)
    elif callable(_fn) and _name.startswith("gen_"):
        globals()[_name] = _wrap_gen_func(_fn)


In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if hasattr(r, "df"):
        r = r.df
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: dedup_redundant_filter ===

# L1 smoke – generated
try:
    _r = gen_dedup_redundant_filter(FIX_DEDUP_REDUNDANT_FILTER_DF_CELLS, FIX_DEDUP_REDUNDANT_FILTER_DF_CROSS_CELLS)
    print("✅ L1 smoke gen_dedup_redundant_filter: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_dedup_redundant_filter: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_dedup_redundant_filter(FIX_DEDUP_REDUNDANT_FILTER_DF_CELLS, FIX_DEDUP_REDUNDANT_FILTER_DF_CROSS_CELLS)
    print("✅ L1 smoke before_dedup_redundant_filter: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_dedup_redundant_filter: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_dedup_redundant_filter(FIX_DEDUP_REDUNDANT_FILTER_DF_CELLS, FIX_DEDUP_REDUNDANT_FILTER_DF_CROSS_CELLS)
    _rg = gen_dedup_redundant_filter(FIX_DEDUP_REDUNDANT_FILTER_DF_CELLS, FIX_DEDUP_REDUNDANT_FILTER_DF_CROSS_CELLS)
    compare(_rb, _rg, "dedup_redundant_filter", check_row_order=True)
except Exception as _e:
    print(f"❌ L2 equivalence dedup_redundant_filter: setup error — {type(_e).__name__}: {_e}")

# L3 — empty typed cross-cells means no redundant rows are removed.
try:
    _rb = before_dedup_redundant_filter(FIX_DEDUP_REDUNDANT_FILTER_DF_CELLS, FIX_DEDUP_REDUNDANT_FILTER_DF_CROSS_CELLS.head(0))
    _rg = gen_dedup_redundant_filter(FIX_DEDUP_REDUNDANT_FILTER_DF_CELLS, FIX_DEDUP_REDUNDANT_FILTER_DF_CROSS_CELLS.head(0))
    compare(_rb, _rg, "L3 dedup_redundant_filter empty typed", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 dedup_redundant_filter empty typed: {type(_e).__name__}: {_e}")

# L3 — a redundant row should remove the matching index_ from df_cells.
try:
    cross = FIX_DEDUP_REDUNDANT_FILTER_DF_CROSS_CELLS.with_columns(pl.Series("redundant", [True, False]))
    _rb = before_dedup_redundant_filter(FIX_DEDUP_REDUNDANT_FILTER_DF_CELLS, cross)
    _rg = gen_dedup_redundant_filter(FIX_DEDUP_REDUNDANT_FILTER_DF_CELLS, cross)
    compare(_rb, _rg, "L3 dedup_redundant_filter removes redundant", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 dedup_redundant_filter removes redundant: {type(_e).__name__}: {_e}")
